# Arbetsflöde: Preprocessing av Satellitdata

Detta skript hanterar huvudsaklig preprocessing för satellitbilder (Landsat och/eller Sentinel-2). Koden itererar igenom mappar med rådata, applicerar molnmasker, klipper ut aktuellt studieområde och beräknar därefter utvalda spektrala index.

**Huvudsakliga steg i processen:**
1. **Konfiguration & inställningar:** Val av sensor och intresseområde. Skriptet initierar även (och kan tömma) målmappar och förbereder en CSV-loggfil.
2. **Laddning av scener:** Skriptet loopar över samtliga nedladdade satellitscener. Datum och grundläggande metadata extraheras från filnamnen/mapparna.
3. **Inläsning och klippning av band:** Relevanta spektrala band (bland annat Röd, Grön, Blå, NIR och SWIR) laddas in och klipps direkt mot valt studieområdes geometri (från shapefiler). Det sparar väsentligt med datorkraft och minne.
4. **Maskering (Cloud Masking):** Ett kvalitetsband (QA_PIXEL för Landsat alt. SCL map för Sentinel) används för att maskera bort otillförlitliga pixlar som moln, molnskugga, och snö.
5. **Skalning (Rescaling):** En omräkning från de lagrade pixelvärdena till reella reflektansvärden (Surface Reflectance) via specifika "scale factors", innan vidare matematiska operationer genomförs.
6. **Beräkning och export:** Systemet beräknar efterfrågade index (t.ex. NDVI, MNDWI). En GeoTIFF genereras för varje index per satellitscen, och information rörande scenens totala molnprocent lagras i loggfilen som används i senare analysskript.

*Obs: Detta skript körs separat för varje respekive studieområde genom att justera konfigurationsvariablerna `sensor` och `area` i koden nedan.*

In [ ]:
import glob
import os
import csv
import rioxarray as rxr
import rasterio as rio
import numpy as np
import geopandas as gdp
from preprocessing import qa_mask, scl_mask

from config import (
    SEA_POLYGON_PATH,
    STUDY_AREA_PATHS,
    build_preprocessing_satellites,
    print_satellite_setup,
)

In [ ]:
# Definiera sensor och namn för studieområde till filutskrift
sensor = "sentinel2"  # "landsat" eller "sentinel2"
area = "TAM"  # OBS: byt område här vid körning

# Läs in studieområdespolygoner från config.py
study_areas = {code: gdp.read_file(path) for code, path in STUDY_AREA_PATHS.items()}
study_area = study_areas[area]

# OBS: används endast i Gunnarsmaren för att maska bort havsområden
sea_polygon = gdp.read_file(SEA_POLYGON_PATH)

# Bygg satellitkonfiguration (input/output + band) från config.py
satellites = build_preprocessing_satellites(area, sensor=sensor)

print_satellite_setup(area, satellites, sensor=sensor)

In [ ]:

# Lista med index som ska beräknas
indices = ["NDVI", "NDWI", "MNDWI", "NDMI", "EVI"]

# ==============================================================
# Töm output-mapparna innan ny körning (sätt till True när du vill rensa)
# Kör om för att ut tidigare körningar
# ==============================================================
clean_output = True

# Skapa output-mappar och kontrollera om de redan finns
for sat in satellites.values():
    for index in indices:
        os.makedirs(os.path.join(sat["output"], index), exist_ok=True)

# Skapa CSV-logg med header för molnprocent (en per studieområde)
for sat_name, sat in satellites.items():
    log_path = os.path.join(sat["output"], f"scene_log_{area}.csv")
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["satellite", "scene_name", "year", "month", "day", "cloud_pct"])

# Töm output-mapparna innan ny körning (om clean_output är True)
# OBS: Raderar endast filer för det aktuella studieområdet (area)
if clean_output:
    confirm = input(f"Är du säker på att du vill tömma output-mapparna för {area}? (y/n): ")
    if confirm.lower() == "y":
        for sat in satellites.values():
            for index in indices:
                index_dir = os.path.join(sat["output"], index)
                for tif_path in glob.glob(os.path.join(index_dir, f"*_{area}_*.tif")):
                    os.remove(tif_path)
        print(f"Output-mappar tömda för {area}.")
    else:
        print("Avbröt – inga filer raderades.")

<details>
<summary><strong>Referenstabeller – Landsat band designations, spektrala index, QA_PIXEL</strong></summary>

> ##### Band designations Landsat missions:

| Band | Landsat 4–7 | Våglängd (µm) | Landsat 8–9 | Våglängd (µm) | Upplösning |
| :-: | :- | :-: | :- | :-: | :-: |
| 1     | Blue                      | 0.45–0.52         | Coastal / Aerosol | 0.43–0.45     | 30 m |
| 2     | Green                     | 0.52–0.60         | Blue              | 0.45–0.51     | 30 m |
| 3     | Red                       | 0.63–0.69         | Green             | 0.53–0.59     | 30 m |
| 4     | NIR                       | 0.76/0.77–0.90    | Red               | 0.64–0.67     | 30 m |
| 5     | SWIR 1                    | 1.55–1.75         | NIR               | 0.85–0.88     | 30 m |
| 6     | Thermal Infrared          | 10.40–12.50       | SWIR1             | 1.57–1.65     | 30 m*|
| 7     | SWIR 2                    | 2.08/2.09–2.35    | SWIR 2            | 2.11–2.29     | 30 m |
| 8     | Panchromatic (Landsat 7)  | 0.52–0.90         | Panchromatic      | 0.50–0.68     | 15 m |
| 9     | —                         | —                 | Cirrus            | 1.36–1.38     | 30 m |
| 10    | —                         | —                 | TIRS 1            | 10.60–11.19   | 30 m*|
| 11    | —                         | —                 | TIRS 2            | 11.50–12.51   | 30 m*|

###### *resampled in at least one mission
###### Source: https://www.usgs.gov/faqs/what-are-band-designations-landsat-satellites


> ##### Specral indicies and application:

| Index | Equation | Usage |
| :-: | :- | :- |
| NDVI   | (NIR − Red) / (NIR + Red)              | Vegatation health and biomass (greenness) |
| NDWI   | (Green − NIR) / (Green + NIR)          | Delineating and monitoring content changes in surface water |
| MNDWI  | (Green − SWIR1) / (Green + SWIR1)      | Enhancement of open water features |
| NDMI   | (NIR − SWIR1) / (NIR + SWIR1)          | Sensitive to the moisture levels in vegetation |
| NDVIre | (NIR − RedEdge) / (NIR + RedEdge)      | Estimating vegetation health using the red-edge band |

###### Source: https://pro.arcgis.com/en/pro-app/latest/arcpy/spatial-analyst/bai.htm; https://pro.arcgis.com/en/pro-app/latest/help/data/imagery/indices-gallery.htm

> ##### QA_PIXEL band – bit definitions (Landsat Collection 2 Level-2)
> 
| Bit(s) | mask_type argument | Description | Notes |
| :-: | :- | :- | :- |
| 0 | `"fill"` | Fill / no-data pixel | |
| 1 | `"dilated"` | Dilated cloud | |
| 2 | `"cirrus"` | Cirrus (single bit) | Unused in Landsat 4–7 |
| 3 | `"cloud"` | Cloud | |
| 4 | `"shadow"` | Cloud shadow | |
| 5 | `"snow"` | Snow / ice | |
| 6 | `"clear"` | Clear pixel | |
| 7 | `"water"` | Water | |
| 8–9 | `"high cloud"` | Cloud confidence **high** | bits 8=1, 9=1 |
| 8–9 | `"mid cloud"` | Cloud confidence **medium** | bits 8=0, 9=1 |
| 8–9 | `"low cloud"` | Cloud confidence **low** | bits 8=1, 9=0 |
| 10–11 | `"high shadow"` | Cloud shadow confidence **high** | bits 10=1, 11=1 |
| 10–11 | `"mid shadow"` | Cloud shadow confidence **medium** | bits 10=0, 11=1 |
| 10–11 | `"low shadow"` | Cloud shadow confidence **low** | bits 10=1, 11=0 |
| 12–13 | `"high snow/ice"` | Snow/ice confidence **high** | bits 12=1, 13=1 |
| 12–13 | `"mid snow/ice"` | Snow/ice confidence **medium** | bits 12=0, 13=1 |
| 12–13 | `"low snow/ice"` | Snow/ice confidence **low** | bits 12=1, 13=0 |
| 14–15 | `"high cirrus"` | Cirrus confidence **high** | bits 14=1, 15=1 – unused L4–7 |
| 14–15 | `"mid cirrus"` | Cirrus confidence **medium** | bits 14=0, 15=1 – unused L4–7 |
| 14–15 | `"low cirrus"` | Cirrus confidence **low** | bits 14=1, 15=0 – unused L4–7 |

###### Confidence-bits (8–15) är par: bit N = LSB, bit N+1 = MSB. Värde 00 = ej bestämd, 01 = låg, 10 = låg–medel (*), 11 = hög.
###### Source: https://code.usgs.gov/eros-user-services/processing_landsat_data/decoding-the-landsat-pixel-quality-assessment-band

</details>


In [ ]:
"""
Loopa över satelliter och scener och skapa index per scen.
OBS: Studieområde styrs av variablerna `sensor` och `area` i konfigurationscellen ovan.
"""

# Loopa över satelliter och scener
saved_count = 0
error_count = 0

for sat_name, sat in satellites.items():
    sensor_kind = sat.get("sensor", "landsat")
    b = sat["bands"]

    if sensor_kind == "sentinel2":
        scene_folders = glob.glob(os.path.join(sat["input"], "S2*_MSIL2A_*"))
    else:
        scene_folders = glob.glob(os.path.join(sat["input"], "*_T1"))  # Väljer tier 1

    print(f"\n[{sat_name}] Hittade {len(scene_folders)} scener i {sat['input']}")

    for scene in scene_folders:
        scene_name = os.path.basename(scene)
        print(f"[{sat_name}] Processing {scene_name}")

        try:
            # =============================================================
            # Extrahera datum ur scennamnet, används till loggfilen
            # =============================================================
            if sensor_kind == "sentinel2":
                scene_date_token = scene_name.split("_")[2]
                scene_date = scene_date_token[:8]
            else:
                scene_date = scene_name.split("_")[3]

            scene_year = scene_date[:4]
            scene_month = scene_date[4:6]
            scene_day = scene_date[6:8]

            # =============================================================
            # Hämta band
            # =============================================================
            if sensor_kind == "sentinel2":
                band_blue = glob.glob(
                    os.path.join(scene, "*.SAFE", "GRANULE", "*", "IMG_DATA", "R20m", f"*_{b['blue']}_20m.jp2")
                )[0]
                band_green = glob.glob(
                    os.path.join(scene, "*.SAFE", "GRANULE", "*", "IMG_DATA", "R20m", f"*_{b['green']}_20m.jp2")
                )[0]
                band_red = glob.glob(
                    os.path.join(scene, "*.SAFE", "GRANULE", "*", "IMG_DATA", "R20m", f"*_{b['red']}_20m.jp2")
                )[0]
                band_nir = glob.glob(
                    os.path.join(scene, "*.SAFE", "GRANULE", "*", "IMG_DATA", "R20m", f"*_{b['nir']}_20m.jp2")
                )[0]
                band_swir = glob.glob(
                    os.path.join(scene, "*.SAFE", "GRANULE", "*", "IMG_DATA", "R20m", f"*_{b['swir']}_20m.jp2")
                )[0]
                band_scl = glob.glob(
                    os.path.join(scene, "*.SAFE", "GRANULE", "*", "IMG_DATA", "R20m", f"*_{b['scl']}_20m.jp2")
                )[0]
            else:
                band_blue = glob.glob(os.path.join(scene, f"*_{b['blue']}.TIF"))[0]
                band_green = glob.glob(os.path.join(scene, f"*_{b['green']}.TIF"))[0]
                band_red = glob.glob(os.path.join(scene, f"*_{b['red']}.TIF"))[0]
                band_nir = glob.glob(os.path.join(scene, f"*_{b['nir']}.TIF"))[0]
                band_swir = glob.glob(os.path.join(scene, f"*_{b['swir']}.TIF"))[0]
                band_qa = glob.glob(os.path.join(scene, "*_QA_PIXEL.TIF"))[0]

            # =============================================================
            # Läs in studieområde och ändra CRS så att den matchar bandens CRS
            # =============================================================
            crs = rio.open(band_red).crs
            study_area_clip = study_area.to_crs(crs).geometry

            # =============================================================
            # Skapa dataarrays för varje band och klipp till studieområdet
            # =============================================================
            blue = rxr.open_rasterio(band_blue).rio.clip(study_area_clip, from_disk=True)
            green = rxr.open_rasterio(band_green).rio.clip(study_area_clip, from_disk=True)
            red = rxr.open_rasterio(band_red).rio.clip(study_area_clip, from_disk=True)
            nir = rxr.open_rasterio(band_nir).rio.clip(study_area_clip, from_disk=True)
            swir = rxr.open_rasterio(band_swir).rio.clip(study_area_clip, from_disk=True)

            if sensor_kind == "sentinel2":
                scl = rxr.open_rasterio(band_scl).rio.clip(study_area_clip, from_disk=True)
            else:
                qa = rxr.open_rasterio(band_qa).rio.clip(study_area_clip, from_disk=True)

            # =============================================================
            # Maska bort havsområden (används endast för Gunnarsmaren)
            # =============================================================
            if area == "GM":
                sea_clip = sea_polygon.to_crs(crs).geometry
                if sensor_kind == "sentinel2":
                    blue, green, red, nir, swir, scl = [
                        band.rio.clip(sea_clip, invert=True, drop=False)
                        for band in (blue, green, red, nir, swir, scl)
                    ]
                else:
                    blue, green, red, nir, swir, qa = [
                        band.rio.clip(sea_clip, invert=True, drop=False)
                        for band in (blue, green, red, nir, swir, qa)
                    ]

            # =============================================================
            # Skala reflektans och skapa masker för kontaminerade pixlar
            # =============================================================
            if sensor_kind == "sentinel2":
                VALID_MIN_DN, VALID_MAX_DN = 1000, 65534 # Exkludera 0 (NoData) och 65535 (mättat)
                BOA_OFFSET = -1000
                SCALE      = 1 / 10000

                # Filtrera NoData och mättade pixlar, applicera offset och scale
                blue, green, red, nir, swir = [
                    band.astype("float32").where((band > VALID_MIN_DN) & (band < VALID_MAX_DN))
                    for band in (blue, green, red, nir, swir)
                ]

                blue, green, red, nir, swir = [
                    (band + BOA_OFFSET) * SCALE
                    for band in (blue, green, red, nir, swir)
                ]

                fill_arr = (scl.data == 0)
                contam_arr = scl_mask(scl.data)
            else:
                VALID_MIN, VALID_MAX = 7273, 43636
                SCALE, OFFSET = 0.0000275, -0.2

                blue, green, red, nir, swir = [
                    band.where((band >= VALID_MIN) & (band <= VALID_MAX)) * SCALE + OFFSET
                    for band in (blue, green, red, nir, swir)
                ]

                fill_arr = qa_mask(qa.data, "fill")  # Pixlar utanför polygonen
                contam_arr = (  # Kontaminerade pixlar: moln, skugga, snö
                    qa_mask(qa.data, "dilated")
                    | qa_mask(qa.data, "cirrus")
                    | qa_mask(qa.data, "cloud")
                    | qa_mask(qa.data, "shadow")
                    | qa_mask(qa.data, "snow")
                )

            qa_mask_arr = fill_arr | contam_arr  # Kombinerad mask för bandmaskning

            # =============================================================
            # Beräkna molntäckning endast inom polygonen (exkluderar fill-pixlar)
            # =============================================================
            valid_pixels = int(np.sum(~fill_arr))
            cloud_pixels = int(np.sum(contam_arr & ~fill_arr))
            cloud_pct = int((cloud_pixels / valid_pixels) * 100) if valid_pixels > 0 else 100
            print(f"    Molntäckning inom polygon: {cloud_pct}%  ({cloud_pixels}/{valid_pixels} pixlar)")

            def apply_mask(band):
                return band.where(~np.broadcast_to(qa_mask_arr, band.shape))

            blue, green, red, nir, swir = [apply_mask(band) for band in (blue, green, red, nir, swir)]

            # =============================================================
            # Beräkna index
            # =============================================================
            evi = 2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))
            
            calc_index = {
                "NDVI": (nir - red) / (nir + red),
                "NDWI": (green - nir) / (green + nir),
                "MNDWI": (green - swir) / (green + swir),
                "NDMI": (nir - swir) / (nir + swir),
                "EVI": evi
            }

            # =============================================================
            # Sparar filer till float32 och ser till att null-värden är NaN
            # =============================================================
            for index_name, result in calc_index.items():
                result = result.astype("float32")
                result.rio.write_nodata(np.nan, inplace=True)

                out_path = os.path.join(
                    sat["output"], index_name, f"{scene_name}_{area}_{index_name}_CLD{cloud_pct}.tif"
                )
                if os.path.exists(out_path):
                    os.remove(out_path)
                result.rio.to_raster(out_path, compress="LZW")
                print(f"    Saved {index_name} → {out_path}")
                saved_count += 1

            # =============================================================
            # Skriver ut CSV-logg (en rad per scen, en fil per studieområde)
            # =============================================================
            log_path = os.path.join(sat["output"], f"scene_log_{area}.csv")
            with open(log_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([sat_name, scene_name, scene_year, scene_month, scene_day, cloud_pct])

        except Exception as e:
            print(f"    FEL för {scene_name}: {e}")
            error_count += 1

print(f"\n=== Klart! Sparade {saved_count} filer, {error_count} fel ===")

#### Snabb kontroll av skalade band om de ger förväntade värden

In [ ]:
# Kontrollera skalade band
for band_name, band in [("Blue", blue), ("Green", green), ("Red", red), ("NIR", nir), ("SWIR", swir)]:
    print(f"{band_name} - max: {float(band.max()):.3f}, min: {float(band.min()):.3f}, medel: {float(band.mean()):.3f}")

# Kontrollera index
ndvi  = (nir - red)   / (nir + red)
mndwi = (green - swir) / (green + swir)
ndmi  = (nir - swir)  / (nir + swir)
evi   = 2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))

for index_name, index in [("NDVI", ndvi), ("MNDWI", mndwi), ("NDMI", ndmi), ("EVI", evi)]:
    print(f"{index_name} - max: {float(index.max()):.3f}, min: {float(index.min()):.3f}, medel: {float(index.mean()):.3f}")

In [ ]:
areas = ["GRM", "GM", "LF", "GS", "AM", "TAM"]

for area in areas:
    for index in ["NDVI", "NDWI", "MNDWI", "NDMI", "EVI"]:
        files = glob.glob(
            rf"D:\Masteruppsats\Sentinel\{'reference' if area in ['AM', 'TAM'] else 'object'}_processed\{index}\*_{area}_{index}_*.tif"
        )
        if not files:
            continue
        test = rxr.open_rasterio(files[0]).squeeze()
        max_val = float(test.max(skipna=True))
        min_val = float(test.min(skipna=True))
        ok = "✓" if -1 <= min_val and max_val <= 1 else "✗"
        print(f"{ok} {area} {index}: max={max_val:.3f}, min={min_val:.3f}")